<a href="https://colab.research.google.com/github/Akshaya200722/AGENTIC_AI/blob/main/4_RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q sentence-transformers faiss-cpu ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 25.9 MB/s eta 0:00:00


In [3]:
import os
from google.colab import files

os.makedirs("Data", exist_ok=True)

uploaded = files.upload()

for filename in uploaded.keys():
    os.rename(filename, os.path.join("Data", filename))

print("Files uploaded successfully:")
print(os.listdir("Data"))

Saving Generative_AI.txt to Generative_AI.txt
Saving NLP.txt to NLP.txt
Saving Deep_Learning.txt to Deep_Learning.txt
Saving ML.txt to ML.txt
Files uploaded successfully:
['ML.txt', 'NLP.txt', 'Deep_Learning.txt', 'Generative_AI.txt']


In [4]:
import os
import numpy as np
import faiss
import ollama
from sentence_transformers import SentenceTransformer

print("Loading embedding model...")

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")


Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [5]:
# --------------------------------------
# Read All Documents
# --------------------------------------

folder = "Data"

documents = []

for filename in os.listdir(folder):

    if filename.endswith(".txt"):

        path = os.path.join(folder, filename)

        with open(path, "r", encoding="utf-8") as file:
            text = file.read()

        documents.append((filename, text))

print("Documents loaded:", len(documents))

for filename, text in documents:
    print(f"- {filename} ({len(text)} characters)")

Documents loaded: 4
- ML.txt (2310 characters)
- NLP.txt (1354 characters)
- Deep_Learning.txt (1454 characters)
- Generative_AI.txt (1601 characters)


In [6]:
# --------------------------------------
# Chunk Documents
# --------------------------------------

chunk_size = 200

chunks = []
metadata = []

for filename, text in documents:

    for i in range(0, len(text), chunk_size):

        chunk = text[i:i + chunk_size]

        chunks.append(chunk)
        metadata.append(filename)

print("\nTotal Chunks:", len(chunks))


Total Chunks: 36


In [7]:
# --------------------------------------
# Create Embeddings
# --------------------------------------

embeddings = model.encode(chunks)

embeddings = np.array(embeddings).astype("float32")

print("Embeddings created successfully!")
print("Embedding shape:", embeddings.shape)

Embeddings created successfully!
Embedding shape: (36, 384)


In [8]:
# --------------------------------------
# Create FAISS Index
# --------------------------------------

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Vector database ready!")
print("Number of vectors:", index.ntotal)

Vector database ready!
Number of vectors: 36


In [9]:
# --------------------------------------
# Ask Question
# --------------------------------------

question = input("\nAsk your question: ")

question_embedding = model.encode([question])

question_embedding = np.array(question_embedding).astype("float32")

# --------------------------------------
# Retrieve Top 3 Chunks
# --------------------------------------

k = 3

distances, indices = index.search(question_embedding, k)

context = ""

print("\nRetrieved Chunks")
print("=" * 60)

for rank, idx in enumerate(indices[0]):

    print(f"\nRank {rank + 1}")
    print("Source:", metadata[idx])
    print(chunks[idx])
    print()

    context += chunks[idx] + "\n"


Ask your question: What is sentiment analysis?

Retrieved Chunks

Rank 1
Source: NLP.txt
ent analysis determines the emotional tone or opinion expressed in text. It is commonly used to identify whether a review or comment is positive, negative, or neutral.

Machine Translation

Machine tr


Rank 2
Source: NLP.txt
sification

Text classification assigns text to predefined categories. Examples include spam detection, sentiment analysis, topic classification, and intent classification.

Sentiment Analysis

Sentim


Rank 3
Source: ML.txt
d and process human language. NLP is used in applications such as chatbots, machine translation, sentiment analysis, and text summarization.

Generative AI

Generative AI refers to artificial intellig

